# Unit 10 exercises: model, stack, solve, and race a plan every tick

Model predictive control plans the next few seconds of steering every tick, applies the first command, and plans again.
These exercises build its pieces one at a time and check each where the answer is known: the error model and its linearisation, the stacked horizon, the least-squares plan and its weights, and the lag, all race-free.
Then they time the solve, put your own MPC inside the shipped agent and race it, compare it honestly with pure pursuit on a new track, and try shorter horizons.
The Challenges try two changes the lesson did not: a terminal cost, and a horizon that depends on speed.
Every number here is a number about racekit's **carla-like car**: a model, not the truth.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.
Every race uses `carla_like(noise=True)` with the stated seeds and `MPC_BUDGET_MS = None` (`NO_TIMING`: no wall clock inside a race, as in the lesson); an MPC race on a `test-circuit` track takes several seconds, and the exercises keep the number of races small.
**Never time anything while other programs are racing**: run the timing exercise (Exercise 5) on its own.

In [ ]:
import math
import time
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_09, UNIT_10 = AGENTS / "unit-09.py", AGENTS / "unit-10.py"
workdir = Path(mkdtemp(prefix="roar-unit-10-ex-"))
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
WHEELBASE = 2.875  # m, L (Unit 3)
MAX_STEER = math.radians(66.0)  # rad: the steering gain, measured (Unit 8), rounded as the agents round it
U_MAX = math.tan(MAX_STEER) / WHEELBASE  # 1/m: the tightest curvature full lock drives
A_THROTTLE = 6.0  # m/s^2 at full throttle (Unit 3's model)
WEIGHTS = (0.3, 20.0, 1000.0)  # (q_e, q_psi, r): agents/unit-10.py's MPC_Q_E, MPC_Q_PSI, MPC_R_DU
NO_TIMING = {"MPC_BUDGET_MS": None}  # every MPC race here: the horizon never depends on the machine's load

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
MEASURED_MU_G = {
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
}


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


def on_track(source, track_name, control, name, **constants):
    # A submission with this track's line and measured grip written in (Unit 9's idiom), plus any constants.
    return variant(source, name, LINE_CONTROL=tuple(float(value) for value in control),
                   MU_G=MEASURED_MU_G[track_name], **constants)


# Unit 9's line-relative clean rule.
def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


# The lesson's reference along the line (section 2), read from an agent's own attributes.
def horizon_reference(agent, i, speed, n):
    # The profile speed (capped by full throttle) and the line's curvature at the distance the car will have
    # travelled after t + 1 ticks, t = 0 .. n - 1, starting from segment i.
    # (The agent also stops the walk after one lap; with n <= 80 ticks it never gets that far, so the two agree.)
    count = len(agent.profile)
    v_ref, k_ref = np.empty(n), np.empty(n)
    s_need, k, travelled = 0.0, i, 0.0
    v = max(speed, 1.0)
    for t in range(n):
        s_need += v * DT
        while travelled < s_need:
            travelled += agent.segment_length[k % count]
            k += 1
        v_ref[t] = max(min(agent.profile[k % count], speed + A_THROTTLE * DT * (t + 1)), 1.0)
        k_ref[t] = agent.curvature[k % count]
        v = v_ref[t]
    return v_ref, k_ref

## Exercise 1 - The error model, and where its linearisation fails

In the path frame, with a = Δt·v and the line's curvature κ, one Euler tick of the model **before** linearising is

e' = e + a sin ψ,  ψ' = ψ + a (u − κ cos ψ / (1 − κ e)),

and one tick of the **linearised** model is

e' = e + a ψ,  ψ' = ψ + a (u − κ).

Write:

- `linear_step(e, psi, u, a, kappa)` and `nonlinear_step(e, psi, u, a, kappa)`: each returns the tuple (e', ψ').

Use a corner of radius 25 m (κ = 0.04 1/m) at 15 m/s (a = Δt·15 = 0.75 m), and the command u = κ (the line's own curvature) throughout.

Define:

- `linear_step`, `nonlinear_step`: the functions;
- `one_tick_e_gap`: a dict from ψ₀ to |e'_linear − e'_nonlinear| after **one** tick from (e, ψ) = (0.0, ψ₀), for ψ₀ = 0.02, 0.1, 0.3 and 0.6 rad;
- `dropped_term`: κ·e for e = 0.5 m, the term the linearisation drops from 1 − κe;
- `blind_gaps`: a dict from ψ₀ to |e_linear − e_nonlinear| after **40** ticks (2 s) from (e, ψ) = (0.5, ψ₀), each model rolled from its own last state, for ψ₀ = 0.02 and 0.3;
- `fig`, `ax`: e against the tick for both models from (0.5, 0.3) over the 40 ticks, with a legend, axis labels and a title;
- `when_it_fails`: two sentences: when is the linear model a good model, and why does an MPC that plans again every tick need it to be good only for a short time near the line?

In [ ]:
def linear_step(e, psi, u, a, kappa):
    return None  # TODO: (e', psi')


def nonlinear_step(e, psi, u, a, kappa):
    return None  # TODO: (e', psi')


one_tick_e_gap = {}  # TODO
dropped_term = None  # TODO
blind_gaps = {}  # TODO
fig, ax = None, None  # TODO
when_it_fails = "TODO"

## Exercise 2 - Stack the horizon, two ways

Over m ticks the errors x = (e_1, ψ_1, e_2, ψ_2, …, e_m, ψ_m) of the linearised model are x = G u + h.

Write:

- `loop_build(e, psi, a, kappa)`: G (2m × m) and h (2m) built one tick at a time, as the lesson's §3 did: the state (e, ψ) and the effect of every command so far are both multiplied by A_t = [[1, a_t], [0, 1]] each tick, the state also gets (0, −a_t κ_t), and command t's column starts as (0, a_t);
- `stack_horizon(e, psi, a, kappa)`: the closed form, with S = `np.cumsum(a)`: the ψ rows of G are a_j for j ≤ t (`np.tril(np.tile(a, (m, 1)))`), the e rows a_j (S_t − S_j) for j ≤ t, h_ψ = ψ − G_ψ κ and h_e = e + S ψ − G_e κ, interleaved e, ψ, e, ψ, ….

Test them on m = 30, `a = 0.05 * (12.0 + 8.0 * np.sin(0.2 * np.arange(30)))`, `kappa = 0.03 * np.cos(0.15 * np.arange(30))`, from (e, ψ) = (0.6, −0.04).

Define:

- `loop_build`, `stack_horizon`: the functions;
- `a_test`, `kappa_test`: the two arrays above;
- `same_G`, `same_h`: `np.allclose` of the two builds' G and of their h, with `rtol=1e-12, atol=1e-12`;
- `u_test`: `0.01 * np.sin(0.3 * np.arange(30))`, and `rollout_gap`: the largest |entry| of (G u_test + h) minus the errors you get by rolling Exercise 1's `linear_step` forward 30 ticks with u_test;
- `causal`: `True` when every entry of G in rows 2t and 2t + 1 and column j > t is exactly 0;
- `loop_ms`, `closed_ms`: the mean time of one build of each (ms), from 200 builds timed with `time.perf_counter()`;
- `why_lower_triangular`: one sentence: what does a zero above the diagonal of G say about the commands and the errors?

In [ ]:
def loop_build(e, psi, a, kappa):
    return None, None  # TODO: (G, h)


def stack_horizon(e, psi, a, kappa):
    return None, None  # TODO: (G, h)


a_test, kappa_test = None, None  # TODO
same_G, same_h = None, None  # TODO
u_test, rollout_gap = None, None  # TODO
causal = None  # TODO
loop_ms, closed_ms = None, None  # TODO
why_lower_triangular = "TODO"

## Exercise 3 - The least-squares plan, and what the weights do

The horizon cost is J(u) = Σ_{t=1}^{m} (q_e e_t² + q_ψ ψ_t²) + r Σ_{t=0}^{m−1} (u_t − u_{t−1})², with u_{−1} the last command applied.

Write:

- `least_squares_plan(G, h, u_last, weights=WEIGHTS)`: the plan u minimising J, from **one** `np.linalg.lstsq` call on the stacked rows [W G; √r D] and right-hand side [−W h; √r d], where W's diagonal is (√q_e, √q_ψ) repeated m times (`np.tile`), D = `np.eye(m) - np.eye(m, k=-1)` and d = (u_last, 0, …, 0);
- `horizon_cost(e, psi, u, a, kappa, u_last, weights=WEIGHTS)`: J computed by rolling `linear_step` forward tick by tick (no matrices).

Use Exercise 2's test horizon (from (0.6, −0.04), `a_test`, `kappa_test`) and u_last = 0.

Define:

- `least_squares_plan`, `horizon_cost`: the functions;
- `plan`: the plan with `WEIGHTS`;
- `cost_of_plan`, `cost_of_line`: J of `plan`, and J of the plan u = κ (`kappa_test`: drive the line's curvature and never correct);
- `beats_nudges`: `True` when `cost_of_plan` is below J of `plan + 1e-4 * np.sin(k * np.arange(30))` for every k = 1 … 5;
- `largest_change`: a dict from r to the largest |u_t − u_{t−1}| of the plan (including u_0 − u_last), for weights (0.3, 20.0, r) with r = 10.0, 1000.0 and 100000.0;
- `final_error`: a dict from the same r to |e_30|, the planned cross-track error at the end of the horizon;
- `first_command`: `plan[0]` clipped to ±`U_MAX`, and `wheel_angle_deg`: the wheel angle that drives it, atan(L u), in degrees;
- `what_r_does`: two sentences: what does a larger r buy, and what does it cost?

In [ ]:
def least_squares_plan(G, h, u_last, weights=WEIGHTS):
    return None  # TODO: the plan, m commands


def horizon_cost(e, psi, u, a, kappa, u_last, weights=WEIGHTS):
    return None  # TODO: J


plan = None  # TODO
cost_of_plan, cost_of_line = None, None  # TODO
beats_nudges = None  # TODO
largest_change = {}  # TODO
final_error = {}  # TODO
first_command, wheel_angle_deg = None, None  # TODO
what_r_does = "TODO"

## Exercise 4 - The lag, on a toy road

A toy road: straight, then from 30 m a 60 m arc of radius 40 m (κ = 1/40), then straight again.
A toy car drives it at a steady 18 m/s; its errors follow Exercise 1's `nonlinear_step`, and it obeys each command **`car_lag` ticks after it was sent** (the commands in flight wait in a queue, oldest first, all 0 at the start).
It starts at (e, ψ) = (0.8, 0.0).

Write:

- `toy_kappa(distance)`: the road's curvature at a distance along it;
- `mpc_tick(e, psi, a, kappa, queue, u_last, weights=WEIGHTS)`: one MPC tick, as the lesson's: roll `linear_step` through the queued commands (the first `len(queue)` ticks of a and kappa), stack the rest of the horizon with `stack_horizon`, solve with `least_squares_plan`, and return the first command clipped to ±`U_MAX`;
- `toy_drive(car_lag, model_lag, ticks=140, speed=18.0, horizon=40)`: the closed loop; every tick, with `distance` the distance driven so far,
  1. the controller's horizon: a_t = Δt·speed and κ_t = `toy_kappa(distance + speed * DT * (t + 1))`, t = 0 … horizon − 1;
  2. the controller's queue is the **last `model_lag`** commands it sent (oldest first; none when `model_lag` is 0), and u_last is the last command it sent;
  3. the car moves one tick with the command it obeys now (the oldest of the `car_lag` commands in flight; with `car_lag` 0, the command just sent) and κ = `toy_kappa(distance + speed * DT)`;
  4. `distance` grows by speed·Δt.

  It returns the array of e after each tick.

Define:

- `toy_kappa`, `mpc_tick`, `toy_drive`: the functions;
- `worst_error`: a dict from (car_lag, model_lag) to the largest |e| after the first 20 ticks, for (car_lag, model_lag) = (1, 1), (1, 0), (2, 2), (2, 1) and (2, 0);
- `fig`, `ax`: e against time (s) for car_lag = 2 and the three model lags, with a legend, axis labels and a title;
- `toy_versus_car`: two or three sentences: on the toy, how much does it cost the controller to forget some of the lag? The lesson's race with `MPC_LAG = 0` was not clean on the carla-like car: why can a kinematic toy at a steady speed not settle that question?

In [ ]:
def toy_kappa(distance):
    return None  # TODO


def mpc_tick(e, psi, a, kappa, queue, u_last, weights=WEIGHTS):
    return None  # TODO: the first command, clipped


def toy_drive(car_lag, model_lag, ticks=140, speed=18.0, horizon=40):
    return None  # TODO: the array of e after each tick


worst_error = {}  # TODO
fig, ax = None, None  # TODO
toy_versus_car = "TODO"

## Exercise 5 - The tick budget

Run this exercise **on its own**, with no other notebook or program racing: a timing taken while the machine is busy measures the machine.

**The solve, race-free.**
Build an agent on `test-circuit` (`with shim_installed(): Unit10, module_name = load_submission(UNIT_10)`, then `agent = Unit10(load_track("test-circuit").waypoints, None)`), and take `start` = the index of the largest |`agent.curvature`| minus 20.
For each horizon m = 10, 20, 40 and 80: `v_ref, k_ref = horizon_reference(agent, start, agent.profile[start], m)`, then call `mpc_tick(0.8, 0.05, DT * v_ref, k_ref, [0.0], 0.0)` 220 times, timing each call with `time.perf_counter()`.
Drop the first 20 calls (the warm-up).

**The step, in a race.**
Race the shipped agent with no timing, `variant(UNIT_10, "shipped", **NO_TIMING)`, on `test-circuit`, seed 0, `max_seconds=450.0`; `result.step_ms` holds the time of every `step()` call (ms).

Define:

- `solve_p50`, `solve_p99`: dicts from m to the median and the 99th percentile (`np.percentile`) of the kept times (ms);
- `shipped_race`: the race result (Exercises 6 and 8 reuse it), and `shipped_offsets`: `np.array(agent.offsets)`, the shipped line's offsets on `test-circuit`;
- `shipped_clean`: `clean(shipped_race, shipped_offsets)`;
- `step_p50`, `step_p99`: the median and 99th percentile of `step_ms` after the first 50 steps (`check_submission`'s warm-up);
- `warmup_max`: the largest of the first 50;
- `inside_budget`: `True` when `step_p99` is at most 5.0 ms;
- `fig`, `ax`: `solve_p99` against m, with axis labels and a title;
- `why_not_in_a_pool`: two sentences: why can a timing taken while eight races share the machine be wrong when the same races' **times** are right?

In [ ]:
solve_p50, solve_p99 = {}, {}  # TODO
shipped_race, shipped_offsets = None, None  # TODO
shipped_clean = None  # TODO
step_p50, step_p99 = None, None  # TODO
warmup_max = None  # TODO
inside_budget = None  # TODO
fig, ax = None, None  # TODO
why_not_in_a_pool = "TODO"

## Exercise 6 - Your MPC inside the agent

The shipped agent calls its method `mpc(self, yaw, speed)` once a tick and does everything else itself: it clips the first command, applies it, queues it for the lag and keeps it as `mpc_last`.
So a subclass that **replaces** `mpc` with your own code races your MPC with everything else unchanged.

Write `agent_plan(agent, yaw, speed, n, weights=WEIGHTS)`, the plan the shipped `mpc()` computes, from the agent's attributes only:

1. the segment i = `agent.segment_index` and j = (i + 1) mod the number of points; the segment's heading is `math.atan2(agent.ys[j] - agent.ys[i], agent.xs[j] - agent.xs[i])`;
2. e = `agent.cross_track_error`, and ψ = yaw − heading, wrapped to [−π, π) (`(angle + math.pi) % (2 * math.pi) - math.pi`);
3. `v_ref, k_ref = horizon_reference(agent, i, speed, n)` and a = Δt·v_ref;
4. roll `linear_step` through the queued commands `agent.mpc_queue` (tick t uses a[t] and k_ref[t]);
5. `stack_horizon` on the rest of the horizon (a and k_ref from index `len(agent.mpc_queue)` on), and return `least_squares_plan(G, h, agent.mpc_last, weights)`.

Then load the class of Exercise 5's variant, `with shim_installed(): Shipped, _ = load_submission(variant(UNIT_10, "shipped", **NO_TIMING))`, and subclass it:

```python
class MyMPC(Shipped):
    def mpc(self, yaw, speed):
        return agent_plan(self, yaw, speed, self.horizon)
```

and race it on `test-circuit`, seed 0, `max_seconds=450.0` (`run_race` accepts a class).

Define:

- `agent_plan`: the function, and `MyMPC`: the class;
- `my_race`: the race result;
- `same_time`: `True` when `my_race.total_time` equals `shipped_race.total_time`;
- `largest_offset_gap`: the largest |difference| of the two races' telemetry `offset` columns (m), or `None` if they have different numbers of rows;
- `what_this_checks`: one sentence: what does an identical race tell you that a race-free test of `stack_horizon` does not?

In [ ]:
def agent_plan(agent, yaw, speed, n, weights=WEIGHTS):
    return None  # TODO: the plan


MyMPC = None  # TODO: the class
my_race = None  # TODO
same_time = None  # TODO
largest_offset_gap = None  # TODO
what_this_checks = "TODO"

## Exercise 7 - The honest comparison on `test-circuit-seed1`

The lesson found that at the same f, on `test-circuit`, MPC is no faster than pure pursuit, and that its gain is the tracking margin.
Is that a `test-circuit` accident?
Solve `test-circuit-seed1`'s line (`seed1 = load_track("test-circuit-seed1")`, `seed1_control = solve_line(seed1)`) and race both agents at f = 0.54 with that track's line and measured grip:

- MPC: `on_track(UNIT_10, "test-circuit-seed1", seed1_control, "seed1-mpc", MU_FRACTION=0.54, **NO_TIMING)`;
- pure pursuit: `on_track(UNIT_09, "test-circuit-seed1", seed1_control, "seed1-pursuit")` (Unit 9 ships 0.54);

each with `repeat_races` on `"test-circuit-seed1"`, seeds 0, 1 and 2 (`n=3, seeds=(0, 1, 2)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=450.0`.

Define:

- `seed1_offsets`: `spline_basis(len(seed1)) @ seed1_control`;
- `mpc_results`, `pursuit_results`: the two `RepeatResult` objects;
- `all_clean`: `True` when all six races are clean (against `seed1_offsets`);
- `medians`, `spreads`: dicts `{"mpc": ..., "pursuit": ...}` (s);
- `worst_errors`: a dict `{"mpc": ..., "pursuit": ...}` of the largest `line_error` over each agent's three races (m);
- `mpc_better`: `noise_aware_better(mpc_results.times, pursuit_results.times)`, and `pursuit_better`: the other way round;
- `margin_m`: `worst_errors["pursuit"] - worst_errors["mpc"]`;
- `verdict`: two sentences: does this track tell the same story as the lesson's `test-circuit`? What would you spend the margin on, and by which rule?

In [ ]:
seed1 = load_track("test-circuit-seed1")
seed1_control = None  # TODO
seed1_offsets = None  # TODO
mpc_results, pursuit_results = None, None  # TODO
all_clean = None  # TODO
medians, spreads = {}, {}  # TODO
worst_errors = {}  # TODO
mpc_better, pursuit_better = None, None  # TODO
margin_m = None  # TODO
verdict = "TODO"

## Exercise 8 - How long a horizon?

The shipped agent plans 40 ticks (2 s).
The lesson's §7 showed the solve's cost growing with the horizon; what does a shorter horizon cost on the track?
Race the shipped agent with `MPC_HORIZON = 10` and with `MPC_HORIZON = 20` (each `variant(UNIT_10, ..., MPC_HORIZON=..., **NO_TIMING)`) on `test-circuit`, seed 0, `max_seconds=450.0`.
The 40-tick race is Exercise 5's `shipped_race`.

Define:

- `horizon_races`: a dict from the horizon (10, 20, 40) to its race result;
- `horizon_clean`: a dict from the horizon to `clean(result, shipped_offsets)`;
- `horizon_times`: a dict from the horizon to the race time (s), and `horizon_errors`: to its `line_error` against `shipped_offsets` (m);
- `cost_of_10`: `horizon_times[10] - horizon_times[40]` (s);
- `fig`, `ax`: the race time against Exercise 5's `solve_p50` for the three horizons, one point per horizon, each plotted with its own `label` (for example `"N = 10"`) so that the legend names it, with axis labels and a title;
- `which_horizon`: two sentences: if your machine's budget for the solve were a p99 of 0.5 ms, which horizon would you ship, and what evidence would you still want before trusting it (one seed, one track)?

In [ ]:
horizon_races = {}  # TODO
horizon_clean = {}  # TODO
horizon_times, horizon_errors = {}, {}  # TODO
cost_of_10 = None  # TODO
fig, ax = None, None  # TODO
which_horizon = "TODO"

## Exercise 9 - Challenge: a terminal cost

The horizon cost weighs every tick's error the same.
A common variation pays **more** for the error at the end of the horizon, so that the plan must end close to the line: a **terminal cost**.
Multiply the last state's two weights, q_e and q_ψ at t = m, by a factor `terminal` (in the stacked rows: multiply the last two entries of W's diagonal by √terminal).

Write:

- `terminal_plan(G, h, u_last, terminal, weights=WEIGHTS)`: Exercise 3's `least_squares_plan` with that change (with `terminal = 1.0` it must give the same plan);
- `agent_terminal_plan(agent, yaw, speed, n, terminal, weights=WEIGHTS)`: Exercise 6's `agent_plan`, solving with `terminal_plan`.

The probe record of plan `roar-009` found the shipped MPC's family edge at 0.60: at 0.62 on `test-circuit-seed1` it was **not clean on seeds 0, 3 and 4** (clean on seeds 1 and 2).
Does a terminal cost move that edge?
Make a subclass, as in Exercise 6, of the class loaded from `on_track(UNIT_10, "test-circuit-seed1", seed1_control, "seed1-0.62", MU_FRACTION=0.62, **NO_TIMING)`, whose `mpc` returns `agent_terminal_plan(self, yaw, speed, self.horizon, 10.0)`, and race it on `"test-circuit-seed1"`, seeds 0 and 3, `max_seconds=450.0`.

Define:

- `terminal_plan`, `agent_terminal_plan`: the functions;
- `same_plan_at_1`: `True` when `terminal_plan(G, h, 0.0, 1.0)` equals `least_squares_plan(G, h, 0.0)` within 1e-12 on Exercise 2's test horizon;
- `end_error`: a dict from `terminal` (1.0, 10.0, 100.0) to the planned |e_30| on Exercise 2's test horizon (from (0.6, −0.04), u_last = 0);
- `terminal_races`: a dict from seed (0, 3) to the race result, and `terminal_clean`: a dict from seed to `clean(result, seed1_offsets)`;
- `terminal_errors`: a dict from seed to `line_error(result, seed1_offsets)` (m);
- `moved_the_edge`: two or three sentences: on these two seeds, is the terminal cost clean where the shipped MPC was not? What more would you have to race before you raised `MU_FRACTION` (the margin rule, the family, the transfer)?

In [ ]:
def terminal_plan(G, h, u_last, terminal, weights=WEIGHTS):
    return None  # TODO


def agent_terminal_plan(agent, yaw, speed, n, terminal, weights=WEIGHTS):
    return None  # TODO


same_plan_at_1 = None  # TODO
end_error = {}  # TODO
terminal_races, terminal_clean = {}, {}  # TODO
terminal_errors = {}  # TODO
moved_the_edge = "TODO"

## Exercise 10 - Challenge: plan a distance, not a time

A 40-tick horizon looks 2 s ahead: 90 m at 45 m/s, but only 20 m at 10 m/s, in the slowest corners, where the line turns most.
Try a horizon that looks a fixed **distance** ahead instead: at speed v,

N(v) = round(DISTANCE / (Δt · max(v, 1))), clipped to 10 … 60 ticks, with DISTANCE = 40 m.

Make a subclass, as in Exercise 6, of the class loaded from `on_track(UNIT_10, "test-circuit-seed4", seed4_control, "seed4-distance", **NO_TIMING)` (`seed4_control = solve_line(load_track("test-circuit-seed4"))`; the shipped `MU_FRACTION`, 0.58), whose `mpc` returns `agent_plan(self, yaw, speed, N(speed))`.
Race it with `repeat_races` on `"test-circuit-seed4"`, seeds 0, 1 and 2, `car_factory=lambda: carla_like(noise=True)`, `max_seconds=450.0`.
The shipped agent's times on those seeds (probe record, plan `roar-009` U10.3) are `SHIPPED_SEED4 = [271.10, 271.10, 271.15]`.

Define:

- `distance_horizon(speed)`: N(v), an int;
- `horizons_seen`: a dict from speed (8.0, 15.0, 25.0, 45.0 m/s) to `distance_horizon(speed)`;
- `seed4_offsets`: the line's offsets, and `distance_results`: the `RepeatResult`;
- `distance_clean`: `True` when all three races are clean;
- `distance_median`: the median time (s), and `worst_line_error`: the largest `line_error` of the three (m);
- `distance_better`: `noise_aware_better(distance_results.times, SHIPPED_SEED4)`, and `shipped_better`: the other way round;
- `worth_it`: two or three sentences: is the distance horizon noise-aware better than the shipped 2 s horizon here? Its longest horizon (60 ticks) costs more time per solve than 40 (Exercise 5): would you ship it, and what else would you race first?

In [ ]:
SHIPPED_SEED4 = [271.10, 271.10, 271.15]  # s: agents/unit-10.py on test-circuit-seed4, seeds 0-2 (plan roar-009)


def distance_horizon(speed):
    return None  # TODO: an int


horizons_seen = {}  # TODO
seed4_control = None  # TODO
seed4_offsets = None  # TODO
distance_results = None  # TODO
distance_clean = None  # TODO
distance_median, worst_line_error = None, None  # TODO
distance_better, shipped_better = None, None  # TODO
worth_it = "TODO"